# Laboratorio 6 — Hito 1: carga, integración y calidad

## tl;dr

- Se cargaron 293 videos y 406 comentarios sin modificar los archivos originales.
- Las llaves primarias son completas y únicas; los 406 comentarios se asocian con un video sin pérdida ni expansión.
- El riesgo principal es la cobertura: solo 19 de 293 videos tienen comentarios recolectados.
- Este hito cubre la actividad 1 y los incisos 2.1–2.4.

## Contexto y métodos

**Fuentes:** `youtube_videos.csv` y `youtube_comments.csv`, proporcionados para el Laboratorio 6.

El análisis preserva los archivos crudos y usa los identificadores de YouTube como llaves. Los nombres y *handles* se mantienen como etiquetas visibles, nunca como sustitutos de los ID. Todos los resultados se calculan desde los CSV al ejecutar el notebook.

### Supuestos clave

- Un valor vacío en `like_count_text` representa cero likes mostrados en la interfaz al recolectar los datos.
- Los conteos son una fotografía del momento de recolección, no valores históricos definitivos.
- `reply_count` no permite crear relaciones entre autores porque no identifica quién respondió.
- Un video sin comentarios en el archivo no se considera socialmente aislado: puede ser una ausencia de cobertura.

## Datos

### 1. Preparar el entorno y localizar las fuentes

In [1]:
from pathlib import Path
import json
import re
import unicodedata
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display, Markdown

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)
pd.set_option("display.max_columns", 50)
pd.set_option("display.max_colwidth", 90)
sns.set_theme(style="whitegrid", context="notebook")

def encontrar_raiz(inicio: Path) -> Path:
    for candidato in (inicio, *inicio.parents):
        if (candidato / "youtube_videos.csv").exists() and (candidato / "youtube_comments.csv").exists():
            return candidato
    raise FileNotFoundError("No se encontraron los CSV del laboratorio en el directorio actual ni en sus padres.")

ROOT = encontrar_raiz(Path.cwd().resolve())
TABLE_DIR = ROOT / "outputs" / "tables"
FIGURE_DIR = ROOT / "outputs" / "figures"
GRAPH_DIR = ROOT / "outputs" / "graphs"
PROCESSED_DIR = ROOT / "data" / "processed"
for directorio in (TABLE_DIR, FIGURE_DIR, GRAPH_DIR, PROCESSED_DIR):
    directorio.mkdir(parents=True, exist_ok=True)

BLUE = "#2378B7"
ORANGE = "#E07A1F"
GOLD = "#D7A928"
CHARCOAL = "#263238"
LIGHT_BLUE = "#A9D2EA"

print(f"Raíz del proyecto: {ROOT}")

Raíz del proyecto: C:\Users\ianro\OneDrive\Documentos\Uvg\Octavo Semestre\Data Science\Lab6DataScience


### 2. Cargar los CSV preservando los identificadores

In [2]:
video_id_columns = ["video_id", "channel_id", "channel_handle", "owner_handle"]
comment_id_columns = ["video_id", "comment_id", "channel_id", "author_channel_id", "author_handle"]

videos_raw = pd.read_csv(
    ROOT / "youtube_videos.csv",
    encoding="utf-8-sig",
    dtype={column: "string" for column in video_id_columns},
)
comments_raw = pd.read_csv(
    ROOT / "youtube_comments.csv",
    encoding="utf-8-sig",
    dtype={column: "string" for column in comment_id_columns},
)

assert videos_raw.shape == (293, 20), f"Dimensión inesperada en videos: {videos_raw.shape}"
assert comments_raw.shape == (406, 17), f"Dimensión inesperada en comentarios: {comments_raw.shape}"

display(pd.DataFrame({
    "dataset": ["youtube_videos.csv", "youtube_comments.csv"],
    "filas": [len(videos_raw), len(comments_raw)],
    "columnas": [videos_raw.shape[1], comments_raw.shape[1]],
    "unidad_de_observacion": ["Un video", "Un comentario principal"],
    "llave_primaria": ["video_id", "comment_id"],
}))

,dataset,filas,columnas,unidad_de_observacion,llave_primaria
0,youtube_videos.csv,293,20,Un video,video_id
1,youtube_comments.csv,406,17,Un comentario principal,comment_id


### 3. Unidad de observación y relaciones

| Entidad | Identificador | Relación relevante |
|---|---|---|
| Canal propietario | `channel_id` | Publica uno o más videos. `channel_name` y `channel_handle` son etiquetas. |
| Video | `video_id` | Pertenece a un canal, tiene categoría y fue recuperado por una consulta. |
| Autor | `author_channel_id` | Puede publicar comentarios en uno o más videos. |
| Comentario | `comment_id` | Pertenece a un video mediante `video_id`. |
| Categoría | `category` | Clasificación asignada por YouTube al video. |
| Consulta | `source_query` / `query_hits` | Describe el muestreo; no equivale necesariamente al tema definitivo. |

La relación entre videos y comentarios es **uno a muchos**: un video puede tener varios comentarios, pero cada comentario pertenece a un solo `video_id`.

## Resultados — Actividades 1 y 2.1–2.4

### 4. Normalización conservadora y conversión de conteos

In [3]:
def normalizar_id(series: pd.Series) -> pd.Series:
    # Elimina espacios externos sin cambiar mayúsculas ni el contenido del ID.
    return series.astype("string").str.strip()

def normalizar_etiqueta(series: pd.Series) -> pd.Series:
    # Normaliza Unicode y espacios en nombres visibles, preservando tildes y capitalización.
    def transformar(value):
        if pd.isna(value):
            return pd.NA
        return re.sub(r"\s+", " ", unicodedata.normalize("NFKC", str(value))).strip()
    return series.map(transformar).astype("string")

def parsear_conteo(value, *, vacio_como_cero=False):
    # Convierte enteros, separadores de miles y abreviaturas K/M; inválidos -> pd.NA.
    if pd.isna(value) or str(value).strip() == "":
        return 0 if vacio_como_cero else pd.NA
    text = unicodedata.normalize("NFKC", str(value)).lower().strip()
    text = re.sub(r"\b(vistas?|likes?|me gusta)\b", "", text).strip().replace(" ", "")
    abbreviated = re.fullmatch(r"(\d+(?:[.,]\d+)?)([km])", text)
    if abbreviated:
        number = float(abbreviated.group(1).replace(",", "."))
        multiplier = 1_000 if abbreviated.group(2) == "k" else 1_000_000
        return int(round(number * multiplier))
    if re.fullmatch(r"\d{1,3}(?:[.,]\d{3})+", text):
        return int(text.replace(",", "").replace(".", ""))
    if re.fullmatch(r"\d+", text):
        return int(text)
    return pd.NA

def parsear_lista_json(value):
    if pd.isna(value) or str(value).strip() == "":
        return []
    try:
        parsed = json.loads(str(value))
    except json.JSONDecodeError:
        return []
    return parsed if isinstance(parsed, list) else []

videos = videos_raw.copy()
comments = comments_raw.copy()

for column in ["video_id", "channel_id"]:
    videos[column] = normalizar_id(videos[column])
for column in ["video_id", "comment_id", "channel_id", "author_channel_id"]:
    comments[column] = normalizar_id(comments[column])
for column in ["channel_name", "channel_handle", "owner_handle"]:
    videos[column] = normalizar_etiqueta(videos[column])
for column in ["channel_name", "author_name", "author_handle"]:
    comments[column] = normalizar_etiqueta(comments[column])

videos["query_hits_lista"] = videos["query_hits"].map(parsear_lista_json)
videos["keywords_lista"] = videos["keywords"].map(parsear_lista_json)
videos["publish_datetime_utc"] = pd.to_datetime(videos["publish_date"], errors="coerce", utc=True)
videos["upload_datetime_utc"] = pd.to_datetime(videos["upload_date"], errors="coerce", utc=True)
videos["view_count_from_text"] = videos["view_count_text"].map(parsear_conteo).astype("Int64")
comments["like_count"] = comments["like_count_text"].map(
    lambda value: parsear_conteo(value, vacio_como_cero=True)
).astype("Int64")

parser_tests = {
    "vacío": parsear_conteo(" ", vacio_como_cero=True),
    "separador_coma": parsear_conteo("1,234"),
    "separador_punto": parsear_conteo("1.234"),
    "abreviatura_k": parsear_conteo("1.2K"),
    "abreviatura_m": parsear_conteo("2,5M"),
    "inválido": parsear_conteo("sin dato"),
}
assert parser_tests["vacío"] == 0
assert parser_tests["separador_coma"] == 1234
assert parser_tests["separador_punto"] == 1234
assert parser_tests["abreviatura_k"] == 1200
assert parser_tests["abreviatura_m"] == 2_500_000
assert pd.isna(parser_tests["inválido"])
parser_tests

{'vacío': 0,
 'separador_coma': 1234,
 'separador_punto': 1234,
 'abreviatura_k': 1200,
 'abreviatura_m': 2500000,
 'inválido': <NA>}

### 5. Perfil de calidad

In [4]:
def contar_atipicos_iqr(series: pd.Series):
    if not pd.api.types.is_numeric_dtype(series.dtype) or pd.api.types.is_bool_dtype(series.dtype):
        return pd.NA
    values = pd.to_numeric(series, errors="coerce").dropna()
    if values.empty:
        return 0
    q1, q3 = values.quantile([0.25, 0.75])
    iqr = q3 - q1
    lower, upper = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    return int(((values < lower) | (values > upper)).sum())

def perfilar_dataset(name: str, frame: pd.DataFrame) -> pd.DataFrame:
    rows = []
    for column in frame.columns:
        series = frame[column]
        blank_count = 0
        if pd.api.types.is_string_dtype(series.dtype) or series.dtype == object:
            blank_count = int(series.fillna("").astype(str).str.strip().eq("").sum())
        rows.append({
            "dataset": name,
            "variable": column,
            "tipo": str(series.dtype),
            "faltantes": int(series.isna().sum()),
            "faltantes_pct": round(float(series.isna().mean() * 100), 2),
            "vacios_texto": blank_count,
            "valores_unicos": int(series.nunique(dropna=True)),
            "constante": bool(series.nunique(dropna=False) <= 1),
            "atipicos_iqr": contar_atipicos_iqr(series),
        })
    return pd.DataFrame(rows)

diagnostico_calidad = pd.concat([
    perfilar_dataset("videos", videos_raw),
    perfilar_dataset("comentarios", comments_raw),
], ignore_index=True)

resumen_calidad = pd.DataFrame({
    "dataset": ["videos", "comentarios"],
    "filas": [len(videos_raw), len(comments_raw)],
    "duplicados_exactos": [int(videos_raw.duplicated().sum()), int(comments_raw.duplicated().sum())],
    "llaves_duplicadas": [int(videos_raw["video_id"].duplicated().sum()), int(comments_raw["comment_id"].duplicated().sum())],
    "llaves_nulas": [int(videos_raw["video_id"].isna().sum()), int(comments_raw["comment_id"].isna().sum())],
})
display(resumen_calidad)
display(diagnostico_calidad.query("faltantes > 0 or vacios_texto > 0 or constante == True"))

,dataset,filas,duplicados_exactos,llaves_duplicadas,llaves_nulas
0,videos,293,0,0,0
1,comentarios,406,0,0,0


,dataset,variable,tipo,faltantes,faltantes_pct,vacios_texto,valores_unicos,constante,atipicos_iqr
8,videos,published_time,str,13,4.44,13,80,False,<NA>
9,videos,view_count_text,str,13,4.44,13,259,False,<NA>
10,videos,description_snippet,str,25,8.53,25,236,False,<NA>
14,videos,description,str,26,8.87,26,234,False,<NA>
33,comentarios,like_count_text,str,0,0.00,189,31,False,<NA>
35,comentarios,is_pinned,bool,0,0.00,0,1,True,<NA>
36,comentarios,viewer_rating,float64,406,100.00,0,0,True,0


### 6. Consistencia de IDs, nombres, handles y campos redundantes

In [5]:
def ids_con_multiples_etiquetas(frame, id_column, label_column):
    counts = frame.dropna(subset=[id_column]).groupby(id_column)[label_column].nunique(dropna=True)
    return int((counts > 1).sum())

consistency_checks = pd.DataFrame([
    {"regla": "channel_id -> un channel_name (videos)", "incumplimientos": ids_con_multiples_etiquetas(videos, "channel_id", "channel_name")},
    {"regla": "channel_id -> un channel_handle", "incumplimientos": ids_con_multiples_etiquetas(videos, "channel_id", "channel_handle")},
    {"regla": "author_channel_id -> un author_name", "incumplimientos": ids_con_multiples_etiquetas(comments, "author_channel_id", "author_name")},
    {"regla": "author_channel_id -> un author_handle", "incumplimientos": ids_con_multiples_etiquetas(comments, "author_channel_id", "author_handle")},
    {"regla": "channel_handle == owner_handle", "incumplimientos": int((videos["channel_handle"].fillna("") != videos["owner_handle"].fillna("")).sum())},
    {"regla": "publish_date == upload_date", "incumplimientos": int((videos["publish_date"].fillna("") != videos["upload_date"].fillna("")).sum())},
    {"regla": "fechas de publicación válidas", "incumplimientos": int(videos["publish_datetime_utc"].isna().sum())},
    {"regla": "query_hits parseable como lista", "incumplimientos": int(videos["query_hits_lista"].map(type).ne(list).sum())},
    {"regla": "keywords parseable como lista", "incumplimientos": int(videos["keywords_lista"].map(type).ne(list).sum())},
])

replacement_character_rows = {
    column: int(videos_raw[column].fillna("").astype(str).str.contains("�", regex=False).sum())
    for column in videos_raw.select_dtypes(include=["object", "string"]).columns
    if videos_raw[column].fillna("").astype(str).str.contains("�", regex=False).any()
}
display(consistency_checks)
print("Filas con carácter de reemplazo Unicode:", replacement_character_rows)

,regla,incumplimientos
0,channel_id -> un channel_name (videos),0
1,channel_id -> un channel_handle,0
2,author_channel_id -> un author_name,0
3,author_channel_id -> un author_handle,0
4,channel_handle == owner_handle,0
5,publish_date == upload_date,0
6,fechas de publicación válidas,0
7,query_hits parseable como lista,0
8,keywords parseable como lista,0


Filas con carácter de reemplazo Unicode: {'description': 1}


### 7. Integración mediante `video_id` y cobertura

In [6]:
video_dimension = videos[[
    "video_id", "title", "channel_id", "channel_name", "channel_handle",
    "category", "source_group", "view_count", "publish_datetime_utc"
]].rename(columns={
    "title": "catalog_video_title",
    "channel_id": "catalog_channel_id",
    "channel_name": "catalog_channel_name",
    "channel_handle": "catalog_channel_handle",
    "source_group": "catalog_source_group",
})

comments_integrated = comments.merge(
    video_dimension,
    on="video_id",
    how="left",
    validate="many_to_one",
    indicator=True,
)
matched_comments = int(comments_integrated["_merge"].eq("both").sum())
orphan_comments = int(comments_integrated["_merge"].eq("left_only").sum())
videos_with_comments = int(videos["video_id"].isin(comments["video_id"]).sum())
videos_without_comments = int(len(videos) - videos_with_comments)

assert len(comments_integrated) == len(comments) == 406
assert matched_comments == 406 and orphan_comments == 0
assert comments_integrated["channel_id"].equals(comments_integrated["catalog_channel_id"])

integration_report = pd.DataFrame({
    "métrica": ["Comentarios totales", "Comentarios asociados", "Comentarios huérfanos", "Videos con comentarios recolectados", "Videos sin comentarios recolectados"],
    "valor": [len(comments), matched_comments, orphan_comments, videos_with_comments, videos_without_comments],
})
display(integration_report)

,métrica,valor
0,Comentarios totales,406
1,Comentarios asociados,406
2,Comentarios huérfanos,0
3,Videos con comentarios recolectados,19
4,Videos sin comentarios recolectados,274


### 8. Hallazgos y tratamiento de variables problemáticas

In [7]:
blank_likes = int(comments_raw["like_count_text"].fillna("").astype(str).str.strip().eq("").sum())
issues = pd.DataFrame([
    ["viewer_rating", "406/406 faltantes", "Alta", "Excluir del análisis; conservar en datos crudos."],
    ["is_pinned", "Constante False", "Baja", "No aporta variación; conservar para auditoría."],
    ["published_time", f"{int(videos_raw['published_time'].isna().sum())} faltantes y formato relativo", "Media", "Usar publish_date en UTC para análisis temporal."],
    ["view_count_text", f"{int(videos_raw['view_count_text'].isna().sum())} faltantes", "Baja", "Usar view_count; conservar el texto para auditoría."],
    ["like_count_text", f"{blank_likes} vacíos", "Media", "Convertir vacíos a cero y guardar like_count numérico."],
    ["upload_date", "Coincide con publish_date en todos los registros", "Baja", "Evitar análisis redundante; preservar ambas columnas crudas."],
    ["owner_handle", "Coincide con channel_handle", "Baja", "Usar channel_id como llave y handle solo como etiqueta."],
    ["description", "Una fila contiene �", "Media", "No inventar el carácter perdido; señalar el defecto de origen."],
    ["cobertura de comentarios", f"Solo {videos_with_comments}/293 videos tienen comentarios", "Alta", "No generalizar participación a todo el catálogo."],
], columns=["variable_o_riesgo", "evidencia", "severidad", "tratamiento"])
display(issues)

diagnostico_calidad.to_csv(TABLE_DIR / "diagnostico_calidad.csv", index=False, encoding="utf-8-sig")
assert (TABLE_DIR / "diagnostico_calidad.csv").exists()
print("Exportado:", TABLE_DIR / "diagnostico_calidad.csv")

,variable_o_riesgo,evidencia,severidad,tratamiento
0,viewer_rating,406/406 faltantes,Alta,Excluir del análisis; conservar en datos crudos.
1,is_pinned,Constante False,Baja,No aporta variación; conservar para auditoría.
2,published_time,13 faltantes y formato relativo,Media,Usar publish_date en UTC para análisis temporal.
3,view_count_text,13 faltantes,Baja,Usar view_count; conservar el texto para auditoría.
4,like_count_text,189 vacíos,Media,Convertir vacíos a cero y guardar like_count numérico.
5,upload_date,Coincide con publish_date en todos los registros,Baja,Evitar análisis redundante; preservar ambas columnas crudas.
6,owner_handle,Coincide con channel_handle,Baja,Usar channel_id como llave y handle solo como etiqueta.
7,description,Una fila contiene �,Media,No inventar el carácter perdido; señalar el defecto de origen.
8,cobertura de comentarios,Solo 19/293 videos tienen comentarios,Alta,No generalizar participación a todo el catálogo.


Exportado: C:\Users\ianro\OneDrive\Documentos\Uvg\Octavo Semestre\Data Science\Lab6DataScience\outputs\tables\diagnostico_calidad.csv


### Conclusiones del primer hito

La integración es completa y no produce explosión de filas. Las llaves son utilizables y consistentes. El principal riesgo no es la integridad de la unión, sino la cobertura: solo una selección pequeña de videos tiene comentarios recolectados. Los valores atípicos en conteos se conservan porque representan popularidad real o concentración potencial, no errores demostrados.